In [ ]:
!pip install optimum[onnxruntime] transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 8.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/notification_model/final_model"
ONNX_OUTPUT_PATH = "/content/drive/MyDrive/notification_model/distilbert_onnx"


In [ ]:
import os
os.makedirs(ONNX_OUTPUT_PATH, exist_ok=True)


In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification

model = ORTModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    export=True
)

model.save_pretrained(ONNX_OUTPUT_PATH)

print("ONNX model exported successfully.")


Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


ONNX model exported successfully.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.save_pretrained(ONNX_OUTPUT_PATH)

print("Tokenizer saved successfully.")


Tokenizer saved successfully.


In [ ]:
import json
import os

# Because you trained on priority values
label_list = [1, 2, 3, 4]

label_map = {i: label for i, label in enumerate(label_list)}

with open(os.path.join(ONNX_OUTPUT_PATH, "label_map.json"), "w") as f:
    json.dump(label_map, f)

print("Label map saved successfully.")


Label map saved successfully.


In [ ]:
os.listdir(ONNX_OUTPUT_PATH)


['config.json',
 'model.onnx',
 'tokenizer_config.json',
 'special_tokens_map.json',
 'vocab.txt',
 'tokenizer.json',
 'label_map.json']

In [ ]:
import torch
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model = ORTModelForSequenceClassification.from_pretrained(ONNX_OUTPUT_PATH)
tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

text = "Your order has been shipped."

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Predicted encoded label:", prediction)


Predicted encoded label: 3


In [ ]:
import torch
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model = ORTModelForSequenceClassification.from_pretrained(ONNX_OUTPUT_PATH)
tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

text = "Congratulations! You won a free prize!"

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Predicted encoded label:", prediction)


Predicted encoded label: 0


**QUANTIZATION**

In [ ]:
!pip install onnxruntime onnxruntime-tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.5 MB/s eta 0:00:00


In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

In [ ]:
import os

original_model_path = os.path.join(ONNX_OUTPUT_PATH, "model.onnx")
quantized_model_path = os.path.join(ONNX_OUTPUT_PATH, "model_quantized.onnx")

In [ ]:
quantize_dynamic(
    model_input=original_model_path,
    model_output=quantized_model_path,
    weight_type=QuantType.QInt8
)

print("Quantization complete!")

Quantization complete!


In [ ]:
import os

original_size = os.path.getsize(original_model_path) / (1024 * 1024)
quantized_size = os.path.getsize(quantized_model_path) / (1024 * 1024)

print(f"Original model size: {original_size:.2f} MB")
print(f"Quantized model size: {quantized_size:.2f} MB")

Original model size: 255.53 MB
Quantized model size: 64.23 MB


In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer
import torch
import json
import os

model = ORTModelForSequenceClassification.from_pretrained(
    ONNX_OUTPUT_PATH,
    file_name="model_quantized.onnx"
)

tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

# Load the label map
with open(os.path.join(ONNX_OUTPUT_PATH, "label_map.json"), "r") as f:
    label_map = json.load(f)

text = "Your meeting is scheduled for tomorrow at 10 AM."
inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)
prediction = torch.argmax(outputs.logits, dim=1).item()

# Convert encoded prediction back to original label
predicted_label = label_map[str(prediction)]

print(f"Input text: '{text}'")
print(f"Predicted encoded label: {prediction}")
print(f"Predicted original label: {predicted_label}")

Input text: 'Your meeting is scheduled for tomorrow at 10 AM.'
Predicted encoded label: 2
Predicted original label: 3


In [ ]:
import torch
import json
import os

# Define the OTP text
otp_text = "Your OTP for transaction is 123456. Do not share this code."

# Tokenize the OTP text
inputs_otp = tokenizer(otp_text, return_tensors="pt")

# Get predictions for OTP text
outputs_otp = model(**inputs_otp)
prediction_otp_encoded = torch.argmax(outputs_otp.logits, dim=1).item()
prediction_otp_original = label_map[str(prediction_otp_encoded)]

print(f"Input text (OTP): '{otp_text}'")
print(f"Predicted encoded label (OTP): {prediction_otp_encoded}")
print(f"Predicted original label (OTP): {prediction_otp_original}")

Input text (OTP): 'Your OTP for transaction is 123456. Do not share this code.'
Predicted encoded label (OTP): 3
Predicted original label (OTP): 4


In [ ]:
import torch
import json
import os

# Define the delivery text
delivery_text = "Your package is out for delivery and will arrive within the next hour."

# Tokenize the delivery text
inputs_delivery = tokenizer(delivery_text, return_tensors="pt")

# Get predictions for delivery text
outputs_delivery = model(**inputs_delivery)
prediction_delivery_encoded = torch.argmax(outputs_delivery.logits, dim=1).item()
prediction_delivery_original = label_map[str(prediction_delivery_encoded)]

print(f"\nInput text (Delivery): '{delivery_text}'")
print(f"Predicted encoded label (Delivery): {prediction_delivery_encoded}")
print(f"Predicted original label (Delivery): {prediction_delivery_original}")


Input text (Delivery): 'Your package is out for delivery and will arrive within the next hour.'
Predicted encoded label (Delivery): 3
Predicted original label (Delivery): 4


In [ ]:
import torch
import json
import os

text_free_prize = "Congratulations! You won a free prize!"

inputs_free_prize = tokenizer(text_free_prize, return_tensors="pt")
outputs_free_prize = model(**inputs_free_prize)
prediction_free_prize_encoded = torch.argmax(outputs_free_prize.logits, dim=1).item()
prediction_free_prize_original = label_map[str(prediction_free_prize_encoded)]

print(f"\nInput text: '{text_free_prize}'")
print(f"Predicted encoded label: {prediction_free_prize_encoded}")
print(f"Predicted original label: {prediction_free_prize_original}")


Input text: 'Congratulations! You won a free prize!'
Predicted encoded label: 0
Predicted original label: 1


In [ ]:
import torch
import json
import os

text_seasonal_sale = "Seasonal Sale: Up to 50% off on all items!"

inputs_seasonal_sale = tokenizer(text_seasonal_sale, return_tensors="pt")
outputs_seasonal_sale = model(**inputs_seasonal_sale)
prediction_seasonal_sale_encoded = torch.argmax(outputs_seasonal_sale.logits, dim=1).item()
prediction_seasonal_sale_original = label_map[str(prediction_seasonal_sale_encoded)]

print(f"\nInput text: '{text_seasonal_sale}'")
print(f"Predicted encoded label: {prediction_seasonal_sale_encoded}")
print(f"Predicted original label: {prediction_seasonal_sale_original}")


Input text: 'Seasonal Sale: Up to 50% off on all items!'
Predicted encoded label: 1
Predicted original label: 2


In [ ]:
import torch
import json
import os

text_delivery_low_priority = "Your package is out for delivery and will arrive within the next hour."

inputs_delivery_low_priority = tokenizer(text_delivery_low_priority, return_tensors="pt")
outputs_delivery_low_priority = model(**inputs_delivery_low_priority)
prediction_delivery_low_priority_encoded = torch.argmax(outputs_delivery_low_priority.logits, dim=1).item()
prediction_delivery_low_priority_original = label_map[str(prediction_delivery_low_priority_encoded)]

print(f"\nInput text: '{text_delivery_low_priority}'")
print(f"Predicted encoded label: {prediction_delivery_low_priority_encoded}")
print(f"Predicted original label: {prediction_delivery_low_priority_original}")


Input text: 'Your package is out for delivery and will arrive within the next hour.'
Predicted encoded label: 3
Predicted original label: 4


In [ ]:

import torch
import json
import os
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

# Load the quantized model and tokenizer
model = ORTModelForSequenceClassification.from_pretrained(
    ONNX_OUTPUT_PATH,
    file_name="model_quantized.onnx"
)
tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

# Load the label map
with open(os.path.join(ONNX_OUTPUT_PATH, "label_map.json"), "r") as f:
    label_map = json.load(f)

text_coffee_meeting = "Hey \"Are we still meeting for coffee later today?\""

inputs_coffee_meeting = tokenizer(text_coffee_meeting, return_tensors="pt")
outputs_coffee_meeting = model(**inputs_coffee_meeting)
prediction_coffee_meeting_encoded = torch.argmax(outputs_coffee_meeting.logits, dim=1).item()
prediction_coffee_meeting_original = label_map[str(prediction_coffee_meeting_encoded)]

print(f"\nInput text: '{text_coffee_meeting}'")
print(f"Predicted encoded label: {prediction_coffee_meeting_encoded}")
print(f"Predicted original label: {prediction_coffee_meeting_original}")


Input text: 'Hey "Are we still meeting for coffee later today?"'
Predicted encoded label: 2
Predicted original label: 3


In [ ]:
import torch
import json
import os
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

# Load the quantized model and tokenizer
model = ORTModelForSequenceClassification.from_pretrained(
    ONNX_OUTPUT_PATH,
    file_name="model_quantized.onnx"
)
tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

# Load the label map
with open(os.path.join(ONNX_OUTPUT_PATH, "label_map.json"), "r") as f:
    label_map = json.load(f)

text_dinner_invite = "Hey, are you coming for dinner tonight?"

inputs_dinner_invite = tokenizer(text_dinner_invite, return_tensors="pt")
outputs_dinner_invite = model(**inputs_dinner_invite)
prediction_dinner_invite_encoded = torch.argmax(outputs_dinner_invite.logits, dim=1).item()
prediction_dinner_invite_original = label_map[str(prediction_dinner_invite_encoded)]

print(f"\nInput text: '{text_dinner_invite}'")
print(f"Predicted encoded label: {prediction_dinner_invite_encoded}")
print(f"Predicted original label: {prediction_dinner_invite_original}")


Input text: 'Hey, are you coming for dinner tonight?'
Predicted encoded label: 2
Predicted original label: 3
